# LogiCrisis — Deep Specialist Manager Agents + GRPO Training
**Meta PyTorch OpenEnv Hackathon | Theme #1: Multi-Agent Interactions**

Stack: `Unsloth` 4-bit QLoRA + `TRL GRPOTrainer` + `LogiCrisisEnv`

### What this notebook does
Trains 6 specialist manager agents using GRPO reinforcement learning:
- **Carrier Manager** — route optimization, R1 delivery × 2.0
- **Warehouse Manager** — cold chain protection, R4 × 3.0
- **Customs Broker Manager** — trade corridors + carbon, R3 × 2.5
- **Insurer Manager** — bid market + coalitions, R3 × 2.5
- **Shipper Manager** — CRITICAL cargo triage, R1 × 2.5
- **Geopolitical Analyst Manager** — intelligence + alerts, R3+R7 × 2.0

Each agent learns from real API data (OpenWeather, GDELT, ExchangeRate)
and self-improves via role-weighted rewards.

> **Runtime**: GPU required — Runtime → Change runtime type → T4 GPU (free) or A100
> **T4**: ~45 min | **A100**: ~12 min

## Step 1 — Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl>=0.9.0" datasets accelerate peft matplotlib
!pip install -q fastapi uvicorn pydantic gradio httpx requests

## Step 2 — Load LogiCrisis from GitHub

In [ ]:
import os, sys

REPO_URL = 'https://github.com/SANGRAMLEMBE/logicriasis.git'
LOGICRIASIS_PATH = '/content/logicriasis'

if not os.path.exists(LOGICRIASIS_PATH):
    !git clone {REPO_URL} {LOGICRIASIS_PATH}
else:
    !cd {LOGICRIASIS_PATH} && git pull origin main

sys.path.insert(0, LOGICRIASIS_PATH)
os.chdir(LOGICRIASIS_PATH)
print('Working directory:', os.getcwd())

from environment.tasks import ALL_TASK_IDS
from agents.role_configs import ROLE_CONFIGS
print('Tasks loaded:', ALL_TASK_IDS)
print('Roles loaded:', list(ROLE_CONFIGS.keys()))

## Step 3 — Heuristic baseline (BEFORE training)
Run this once — these are your **before** scores.

In [ ]:
import sys
sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

from inference import run_episode
from environment.tasks import ALL_TASK_IDS

print('=== HEURISTIC BASELINE (no LLM) ===')
baseline_scores = {}
for task_id in ALL_TASK_IDS:
    result = run_episode(task_id, use_llm=False)
    baseline_scores[task_id] = result['score']
    status = 'PASS' if result['passed'] else 'FAIL'
    print(f'  {task_id:<35} score={result["score"]:.4f}  {status}')

avg = sum(baseline_scores.values()) / len(baseline_scores)
print(f'\n  Average heuristic: {avg:.4f}')
print('\n  Save these numbers — this is your BEFORE.')

## Step 4 — Verify role configs and reward weights

In [ ]:
from agents.role_configs import ROLE_CONFIGS
from agents.prompts import get_system_prompt, get_allowed_actions

print('=== SPECIALIST MANAGER CONFIGS ===')
for role, cfg in ROLE_CONFIGS.items():
    weights = cfg['reward_weights']
    top_rewards = sorted(weights.items(), key=lambda x: -x[1])[:3]
    top_str = ' | '.join(f'{k}x{v}' for k, v in top_rewards)
    print(f'\n  [{cfg["title"]}]')
    print(f'    Specialty: {cfg["specialty"]}')
    print(f'    Top rewards: {top_str}')
    print(f'    Allowed actions: {get_allowed_actions(role)}')
    print(f'    API signals: {cfg["api_signals"]}')

## Step 5 — Reward function sanity check (role-specific)

In [ ]:
from training.train import _score_completion

print('=== REWARD FUNCTION TESTS ===')

# Carrier: reroute with good reasoning
carrier_good = '{"action_type": "reroute", "cargo_id": "C001", "route_id": "Mumbai-Pune", "reasoning": "Delhi-Jaipur blocked, rerouting C001 priority=4 via Mumbai-Pune. Late delivery -0.5 beats undelivered -1.0 pts"}'
# Warehouse: deploy cold storage with quality reasoning
warehouse_good = '{"action_type": "deploy_cold_storage", "cargo_id": "C003", "reasoning": "C003 cold_chain deadline in 2 turns, temperature alert. Deploying now prevents R4 spoilage penalty worth 3x normal miss."}'
# Insurer: make a risk-priced bid
insurer_good = '{"action_type": "make_bid", "target_agent": "shipper_0", "bid_price": 1500, "bid_capacity": 10.0, "reasoning": "GDELT severity=3 in region. Risk premium base 1000 + 500 = 1500. Accepted bid earns R3 and 0.25 market_score."}'
# Wait with no reasoning (bad)
bad_wait = '{"action_type": "wait"}'
# Malformed (worst)
malformed = 'I will reroute cargo to Mumbai'

print(f'  Carrier reroute (good):       {_score_completion(carrier_good, role="carrier"):+.3f}  (expect ~+0.65)')
print(f'  Warehouse cold storage:       {_score_completion(warehouse_good, role="warehouse"):+.3f}  (expect ~+0.65)')
print(f'  Insurer bid (good):           {_score_completion(insurer_good, role="insurer"):+.3f}  (expect ~+0.65)')
print(f'  Wait with no reasoning:       {_score_completion(bad_wait, role="carrier"):+.3f}  (expect ~-0.05)')
print(f'  Malformed output:             {_score_completion(malformed):+.3f}  (expect -0.5)')

## Step 6 — Load model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/llama-3-8b-instruct-bnb-4bit'
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded:', MODEL_NAME)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Step 7 — Build multi-role curriculum dataset
512 prompts: 40% L1 (easy) + 40% L2 (medium) + 20% L3 (hard)
All 6 specialist roles represented.

In [ ]:
from training.train import build_curriculum_dataset
from collections import Counter

dataset = build_curriculum_dataset(warmup_samples=512)
print(f'Total prompts: {len(dataset)}')

print('\nRole distribution:')
role_counts = Counter(dataset['role'])
for role, cnt in sorted(role_counts.items()):
    bar = '#' * (cnt // 5)
    print(f'  {role:<25} {cnt:>4}  {bar}')

print('\nSample prompt (carrier, first 600 chars):')
carrier_samples = [r for r in dataset if r['role'] == 'carrier']
if carrier_samples:
    print(carrier_samples[0]['prompt'][:600])

## Step 8 — GRPO Training with role-weighted rewards

**T4**: ~45 min | **A100**: ~12 min

Role reward weights during training:
- Carrier: R1_delivery × 2.0, R5_efficiency × 1.5
- Warehouse: R4_cold_chain × 3.0
- Customs Broker: R3_negotiation × 2.5, R7_carbon × 2.0
- Insurer: R3_negotiation × 2.5, R2_coalition × 2.0
- Shipper: R1_delivery × 2.5
- Geopolitical Analyst: R3 × 2.0, R7 × 2.0

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.train import grpo_reward_fn

grpo_config = GRPOConfig(
    output_dir='/content/logicriasis_outputs',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=50,
    report_to='none',
    max_completion_length=256,
    num_generations=8,
    temperature=0.8,
    seed=42,
    fp16=True,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[grpo_reward_fn],
    args=grpo_config,
    train_dataset=dataset,
)

print('Starting GRPO training with 6 specialist manager roles...')
print('Role-weighted rewards active:')
print('  Carrier: R1x2.0 | Warehouse: R4x3.0 | Broker: R3x2.5')
print('  Insurer: R3x2.5 | Shipper: R1x2.5 | GeoAnalyst: R3+R7x2.0')
trainer.train()
print('Training complete!')

## Step 9 — Save adapters + generate training curves

In [ ]:
import os, shutil
from training.train import save_training_curves

# Save LoRA adapters
SAVE_DIR = '/content/logicriasis_outputs/final'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Adapters saved to {SAVE_DIR}')

# Save training curves to assets/
os.makedirs('assets', exist_ok=True)
save_training_curves(trainer.state.log_history, output_dir='assets')

# Copy adapters to Drive for persistence
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_save = '/content/drive/MyDrive/logicriasis_adapters'
    shutil.copytree(SAVE_DIR, drive_save, dirs_exist_ok=True)
    print(f'Adapters also saved to Drive: {drive_save}')
except Exception as e:
    print(f'Drive save skipped: {e}')

# Commit PNGs to GitHub
import subprocess
try:
    result = subprocess.run(
        ['git', 'config', 'user.email', 'colab@logicriasis.ai'],
        cwd='/content/logicriasis', capture_output=True
    )
    subprocess.run(
        ['git', 'config', 'user.name', 'LogiCrisis Colab'],
        cwd='/content/logicriasis', capture_output=True
    )
    subprocess.run(
        ['git', 'add', 'assets/loss_curve.png', 'assets/reward_curve.png'],
        cwd='/content/logicriasis'
    )
    subprocess.run(
        ['git', 'commit', '-m', 'feat: add GRPO training curves from Colab run'],
        cwd='/content/logicriasis'
    )
    subprocess.run(
        ['git', 'push', 'origin', 'main'],
        cwd='/content/logicriasis'
    )
    print('Training curves committed to GitHub')
except Exception as e:
    print(f'Git push skipped (set up credentials to enable): {e}')

## Step 10 — Enable fast inference on trained model

In [ ]:
import torch, json
from unsloth import FastLanguageModel
from environment.models import AgentAction, ActionType
from agents.prompts import get_system_prompt, build_user_prompt, get_allowed_actions
from agents.role_configs import compute_role_weighted_reward

FastLanguageModel.for_inference(model)
print('Model switched to inference mode')

def generate_action(agent_id: str, obs) -> AgentAction:
    role = obs.role.value
    messages = [
        {'role': 'system', 'content': get_system_prompt(role)},
        {'role': 'user',   'content': build_user_prompt(obs.to_prompt_text())},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=256,
            do_sample=True, temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()
    if response.startswith('```'):
        response = '\n'.join(l for l in response.split('\n')
                              if not l.strip().startswith('```'))

    # Validate action against role's allowed list
    allowed = get_allowed_actions(role)
    try:
        data = json.loads(response)
        action_str = data.get('action_type', 'wait')
        if action_str not in allowed:
            action_str = 'wait'
        return AgentAction(
            agent_id=agent_id,
            action_type=ActionType(action_str),
            cargo_id=data.get('cargo_id'),
            route_id=data.get('route_id'),
            target_region=data.get('target_region'),
            bid_price=data.get('bid_price'),
            bid_capacity=data.get('bid_capacity'),
            target_agent=data.get('target_agent'),
            bid_id=data.get('bid_id'),
            coalition_id=data.get('coalition_id'),
            coalition_members=data.get('coalition_members'),
            coalition_role=data.get('coalition_role'),
            reward_split=data.get('reward_split'),
            reasoning=data.get('reasoning', ''),
        )
    except (json.JSONDecodeError, ValueError):
        return AgentAction(agent_id=agent_id, action_type=ActionType.WAIT,
                           reasoning='parse_error')


def run_task_with_llm(task_id: str) -> dict:
    from environment.tasks import get_task
    task = get_task(task_id)
    env = task.make_env(seed=42)
    observations = env.reset()
    for _ in range(env.world.max_turns):
        actions = {aid: generate_action(aid, obs)
                   for aid, obs in observations.items()}
        result = env.step(actions)
        observations = result.observations
        if result.terminated or result.truncated:
            break
    return task.grade(env)

print('generate_action() and run_task_with_llm() ready')

## Step 11 — Before / After comparison (THE MONEY SLIDE)

Shows trained specialist agents vs heuristic baseline across all 9 tasks.

In [ ]:
from environment.tasks import ALL_TASK_IDS

print('=== GRPO SPECIALIST AGENTS vs HEURISTIC BASELINE ===')
print(f'{"Task":<35} {"Heuristic":>10} {"LLM":>10} {"Delta":>8} {"Status"}')
print('-' * 75)

llm_scores = {}
for task_id in ALL_TASK_IDS:
    result = run_task_with_llm(task_id)
    llm_score = result['score']
    heur_score = baseline_scores.get(task_id, 0.0)
    llm_scores[task_id] = llm_score
    delta = llm_score - heur_score
    status = 'PASS' if result['passed'] else 'FAIL'
    sign = '+' if delta >= 0 else ''
    print(f'{task_id:<35} {heur_score:>10.4f} {llm_score:>10.4f} {sign+f"{delta:.4f}":>8}  {status}')

print('-' * 75)
avg_heur = sum(baseline_scores.values()) / len(baseline_scores)
avg_llm  = sum(llm_scores.values()) / len(llm_scores)
delta    = avg_llm - avg_heur
sign     = '+' if delta >= 0 else ''
print(f'{"AVERAGE":<35} {avg_heur:>10.4f} {avg_llm:>10.4f} {sign+f"{delta:.4f}":>8}')
print(f'\n  Heuristic pass rate: {sum(1 for s in baseline_scores.values() if s >= 0.35)}/9')
print(f'  LLM pass rate:       {sum(1 for s in llm_scores.values() if s >= 0.35)}/9')
print(f'  Average improvement: {sign}{delta:.4f}')

## Step 12 — Before/After bar chart + save

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

tasks = list(ALL_TASK_IDS)
heur_vals = [baseline_scores.get(t, 0) for t in tasks]
llm_vals  = [llm_scores.get(t, 0) for t in tasks]

x = np.arange(len(tasks))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
bars1 = ax.bar(x - width/2, heur_vals, width, label='Heuristic baseline',
               color='#94a3b8', alpha=0.85)
bars2 = ax.bar(x + width/2, llm_vals, width, label='GRPO Specialist Agents',
               color='#3b82f6', alpha=0.85)

ax.axhline(y=0.35, color='#ef4444', linestyle='--', linewidth=1.2, label='Pass threshold (0.35)')
ax.set_xlabel('Task')
ax.set_ylabel('Score')
ax.set_title('LogiCrisis: GRPO Specialist Agents vs Heuristic Baseline')
ax.set_xticks(x)
ax.set_xticklabels([t.replace('_', '\n') for t in tasks], fontsize=8)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, axis='y', alpha=0.3)

fig.tight_layout()
os.makedirs('assets', exist_ok=True)
fig.savefig('assets/before_after.png', dpi=150)
print('Saved assets/before_after.png')

# Commit to GitHub
try:
    subprocess.run(['git', 'add', 'assets/before_after.png'],
                   cwd='/content/logicriasis')
    subprocess.run(['git', 'commit', '-m', 'feat: add before/after comparison chart'],
                   cwd='/content/logicriasis')
    subprocess.run(['git', 'push', 'origin', 'main'],
                   cwd='/content/logicriasis')
    print('Chart committed to GitHub')
except Exception as e:
    print(f'Git push skipped: {e}')

## Step 13 — Show specialist agent reasoning (qualitative)
See how each manager agent reasons differently in the same scenario.

In [ ]:
from environment.tasks import get_task

for demo_task_id in ['capacity_crunch', 'earthquake_relief']:
    print(f'\n{"="*65}')
    print(f'TASK: {demo_task_id}')
    print(f'{"="*65}')

    task = get_task(demo_task_id)
    env = task.make_env(seed=42)
    observations = env.reset()

    for turn_num in range(3):
        print(f'\n--- Turn {turn_num + 1} ---')
        actions = {}
        for agent_id, obs in observations.items():
            action = generate_action(agent_id, obs)
            actions[agent_id] = action
            role = obs.role.value
            print(f'  [{agent_id} | {role}]')
            print(f'    action: {action.action_type.value}')
            if action.cargo_id:         print(f'    cargo: {action.cargo_id}  route: {action.route_id}')
            if action.bid_price:        print(f'    bid: ${action.bid_price}  capacity: {action.bid_capacity}t  target: {action.target_agent}')
            if action.coalition_members: print(f'    coalition: {action.coalition_members}')
            if action.reasoning:        print(f'    reasoning: "{action.reasoning[:120]}"')

        result = env.step(actions)
        observations = result.observations
        print(f'  OTIF: {result.info["otif_percent"]}%')
        if result.terminated or result.truncated:
            break

    grade = task.grade(env)
    print(f'\nFinal: score={grade["score"]:.4f}  {grade["verdict"]}')

## Step 14 — Optional: Launch Gradio demo

In [ ]:
import subprocess, threading

def run_demo():
    subprocess.run(['python', 'demo/app.py'])

t = threading.Thread(target=run_demo, daemon=True)
t.start()
print('Demo starting — look for the Gradio share link above (valid 72h)')